In [1]:
# celelb에서 이미지 읽어오는 Dataloader 만들기
# vae latent shape에 대응하도록 모델 짜기
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from tqdm import tqdm
from diffusers import AutoencoderKL
import os
from utils.utils import visualize
from utils.sample import ddpm_sample, ddim_sample
import torch.nn.functional as F
from ldm.dataset import CocoDataset
from ldm.unet import Unet

In [2]:
from transformers import CLIPTokenizer, CLIPTextModel

model_name = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(model_name)

# 예제 문장을 텐서로 변환
# text_inputs = tokenizer(["This is dream", "This is dream"], return_tensors="pt")
# 텍스트 인코딩
# text_features = text_encoder(**text_inputs)

device = 'cuda'

text_encoder = CLIPTextModel.from_pretrained(model_name).to(device)
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)
unet = Unet(init_resolution=32).to(device)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


resolution :  16
resolution :  8
resolution :  4
resolution :  4
resolution :  4
resolution :  8
resolution :  16
resolution :  32


In [3]:
batch_size = 4
learning_rate = 0.0001
epochs = 1000
total_timesteps = 1000
beta_0 = 0.0001
beta_T = 0.02
sampling_steps = 1000

output_dir = './logs_ldm_condition'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    os.makedirs(f"{output_dir}/valid_imgs/")
    os.makedirs(f"{output_dir}/weights/")

optimizer = torch.optim.Adam(unet.parameters(), lr=learning_rate, betas=(0.9, 0.999))

trainer = {
    'train_losses': [],
    'valid_losses': [],
    'valid_images': [],
}

In [4]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((256, 256), antialias=True),
    transforms.RandomHorizontalFlip(p=0.1),
    transforms.Normalize([0.5], [0.5])
])

dataset_train = CocoDataset("./train.json", tokenizer=tokenizer, transforms=transform)
dataset_valid = CocoDataset("./val.json", tokenizer=tokenizer, transforms=transform, is_valid=True)

dataloader_train = torch.utils.data.DataLoader(
    dataset_train, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=8
)
dataloader_valid = torch.utils.data.DataLoader(
    dataset_valid, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=8
)

100%|██████████| 5000/5000 [00:00<00:00, 1568550.49it/s]


In [5]:
def linear_beta_schedule(timesteps=1000):
    betas = torch.linspace(beta_0, beta_T, steps=timesteps+1)
    alphas = 1. - betas
    alphas_bar = torch.cumprod(alphas, dim=0)

    return betas, alphas, alphas_bar

betas, alphas, alphas_bar = linear_beta_schedule(total_timesteps)
betas, alphas, alphas_bar = betas.to(device), alphas.to(device), alphas_bar.to(device)

def x_t_sample(x_0, timesteps, noise):
    return torch.stack([torch.sqrt(alphas_bar[t])*x_0[idx] + torch.sqrt(1-alphas_bar[t])*noise[idx] for idx, t in enumerate(timesteps)])

def x_t_1_sample(x_t, timesteps, predicted_noise, z):
    moved_mean = torch.stack([x_t[idx] - (1-alphas[t])/(torch.sqrt(1-alphas_bar[t])) * predicted_noise[idx] for idx, t in enumerate(timesteps)])
    return torch.stack([1/torch.sqrt(alphas[t]) * moved_mean[idx] + torch.sqrt(betas[t]) * z[idx] for idx, t in enumerate(timesteps)])

def write(text):
    with open(f'{output_dir}/logs.txt', 'a') as file:
        file.write(text)

In [ ]:
from datetime import datetime
write(f"\n\nTraining start : {datetime.today().strftime('%Y-%m-%d %H:%M')}\n\n")

torch.cuda.empty_cache()

vae.eval()
text_encoder.eval()
for epoch in range(epochs):
    unet.train()
    epoch_loss = 0
    tqdm_bar = tqdm(total=len(dataloader_train), desc="Diffusion Training")
    
    for idx, data in enumerate(dataloader_train):
        optimizer.zero_grad()
        
        x_0 = data['image'].to(device)
        b, c, h, w = x_0.shape
        with torch.no_grad():
            input_ids = data['caption']['input_ids'].squeeze().to(device)
            attention_mask = data['caption']['attention_mask'].squeeze().to(device)
            
            z_0 = vae.encode(x_0)
            z_0 = z_0['latent_dist'].sample() * 0.18215

            text_embed = text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
            del input_ids
            del attention_mask
            del x_0
            torch.cuda.empty_cache()
        
        timesteps = torch.randint(1, 1001, (b,))
        added_noise = torch.randn_like(z_0)
        
        z_t = x_t_sample(z_0, timesteps, added_noise)
        z_t = z_t.to(device)
        timesteps = timesteps.to(device)
        added_noise = added_noise.to(device)

        predicted_noise = unet(z_t, timesteps, text_embed)
        
        loss = F.mse_loss(added_noise, predicted_noise)
        
        loss.backward()
        optimizer.step()
        
        tqdm_bar.update()
        epoch_loss += loss.cpu().detach().item()
        if idx%100==99 and epoch>0:
            trainer['train_losses'].append(epoch_loss/idx)
        
    train_text=f'Epoch {epoch} Train loss - {epoch_loss/len(dataloader_train)}\n'
    write(train_text)
    
    # del loss
    # del predicted_noise
    # del added_noise
    # del timesteps
    # del z_t
    # del text_embed
    # torch.cuda.empty_cache()

    unet.eval()
    valid_loss = 0
    tqdm_bar = tqdm(total=len(dataloader_valid), desc="Diffusion validation")
    with torch.no_grad():
        for idx, data in enumerate(dataloader_valid):
            x_0 = data['image'].to(device)
            b, c, h, w = x_0.shape
            with torch.no_grad():
                input_ids = data['caption']['input_ids'].squeeze().to(device)
                attention_mask = data['caption']['attention_mask'].squeeze().to(device)
                text_embed = text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
                
                z_0 = vae.encode(x_0)
                z_0 = z_0['latent_dist'].sample() * 0.18215
                del x_0
                torch.cuda.empty_cache()
            
            timesteps = torch.randint(1, 1001, (b,))
            added_noise = torch.randn_like(z_0)
            
            z_t = x_t_sample(z_0, timesteps, added_noise)
            z_t = z_t.to(device)
            timesteps = timesteps.to(device)
            added_noise = added_noise.to(device)
    
            predicted_noise = unet(z_t, timesteps, text_embed)
            
            loss = F.mse_loss(added_noise, predicted_noise)
            
            valid_loss += loss.cpu().detach().item()
            tqdm_bar.update()

            if idx==0:
                ddim_timesteps = torch.linspace(1000, 10, 100).int().to(device)
                # Inference Sampling
                x_t = torch.randn_like(z_0).to(device)
                for t in ddim_timesteps:
                    t = t.repeat(b)
                    predict_noise = unet(x_t, t, text_embed)
                    x_t = ddim_sample(x_t, t, predict_noise, alphas_bar[t], alphas_bar[t-10])

                predicted_image = vae.decode(x_t)
                trainer['valid_images'].append(predicted_image['sample'][:4].cpu().detach())
                del ddim_timesteps
                del predicted_image
        
        del loss
        del predicted_noise
        del added_noise
        del timesteps
        del z_t
        torch.cuda.empty_cache()
        trainer['valid_losses'].append(valid_loss/len(dataloader_valid))

    if valid_loss/len(dataloader_valid) <= min(trainer['valid_losses']):
        torch.save(unet.state_dict(), f'{output_dir}/weights/model_{epoch}.pth')
    valid_text=f'Epoch {epoch} Validation loss - {valid_loss/len(dataloader_valid)}\n\n'
    write(valid_text)

    plt.plot(trainer['train_losses'])
    plt.savefig(f'{output_dir}/train_loss.png')
    plt.close()

    plt.plot(trainer['valid_losses'])
    plt.savefig(f'{output_dir}/valid_loss.png')
    plt.close()

    visualize(trainer['valid_images'][-1], epoch=epoch, save=True, output_dir=output_dir)

Diffusion Training:  98%|█████████▊| 28980/29572 [3:51:56<04:40,  2.11it/s]  IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Diffusion Training:  16%|█▋        | 4851/29572 [38:51<3:22:17,  2.04it/s]

In [15]:
import json
import os

with open("./val.json", 'r') as f:
    datas = json.load(f)

print(len(datas))
print(len(os.listdir('/workspace/val2017/')))

vals = set(list(os.listdir('/workspace/val2017/')))

5000
5000


In [16]:
datas[0]

{'caption': 'A black Honda motorcycle parked in front of a garage.',
 'file_name': '000000179765.jpg'}

In [18]:
for d in datas:
    if d['file_name'] not in vals:
        print(d)

In [8]:
[1,2,3,4,3].extend([6,7])

In [10]:
import torch

torch.rand((16,))

tensor([0.2300, 0.1383, 0.8756, 0.3246, 0.1673, 0.0799, 0.1746, 0.2033, 0.4267,
        0.0082, 0.4333, 0.8991, 0.1207, 0.0223, 0.9933, 0.8962])